# GLoVe, FastText and BPE

In [ ]:
!pip install datasets

In [72]:
import numpy as np
import re
from collections import Counter
from datasets import load_dataset

In [20]:
dataset = load_dataset("stanfordnlp/imdb")

Generating unsupervised split: 100%|██████████| 50000/50000 [00:00<00:00, 282995.46 examples/s]


In [108]:
def clean_review(text):
    text = text.lower().replace("<br />", " ")
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_reviews(texts, max_docs=None):
    docs = []
    for text in texts[:max_docs]:
        docs.append(clean_review(text).split())
    return docs


# Start small. Full IMDB is slow for this pure-Python implementation.
docs = tokenize_reviews(dataset["train"]["text"], max_docs=10000)
print(len(docs), "reviews")
print(docs[0][:30])

10000 reviews
['i', 'rented', 'i', 'am', 'curious', 'yellow', 'from', 'my', 'video', 'store', 'because', 'of', 'all', 'the', 'controversy', 'that', 'surrounded', 'it', 'when', 'it', 'was', 'first', 'released', 'in', 'i', 'also', 'heard', 'that', 'at', 'first']


In [109]:
def build_vocab(docs, max_vocab=5000, min_count=5):
    counts = Counter(token for doc in docs for token in doc)
    words = [
        word for word, count in counts.most_common(max_vocab)
        if count >= min_count
    ]
    word_to_id = {word: i for i, word in enumerate(words)}
    id_to_word = {i: word for word, i in word_to_id.items()}
    return word_to_id, id_to_word


def build_cooccurrence(docs, window=5, max_vocab=5000, min_count=5):
    pair_counts = Counter()
    word_to_id, id_to_word = build_vocab(docs, max_vocab=max_vocab, min_count=min_count)

    for doc in docs:
        indexed = [word_to_id[t] for t in doc if t in word_to_id]
        for i, center in enumerate(indexed):
            for j in range(max(0, i - window), min(len(indexed), i + window + 1)):
                if i != j:
                    distance = abs(i - j)
                    pair_counts[(center, indexed[j])] += 1.0 / distance

    print("Vocab size:", len(word_to_id))
    print("Co-occurrence pairs:", len(pair_counts))
    return word_to_id, id_to_word, pair_counts



In [110]:

def glove_train(word_to_id, pair_counts, dim=50, epochs=25, lr=0.03, x_max=100, alpha=0.75, seed=0):
    n = len(word_to_id)
    rng = np.random.default_rng(seed)
    W = rng.normal(0, 0.1, size=(n, dim))
    W_tilde = rng.normal(0, 0.1, size=(n, dim))
    b = np.zeros(n)
    b_tilde = np.zeros(n)

    pairs = list(pair_counts.items())
    for epoch in range(epochs):
        rng.shuffle(pairs)
        total_loss = 0.0
        for (i, j), x_ij in pairs:
            weight = (x_ij / x_max) ** alpha if x_ij < x_max else 1.0
            diff = W[i] @ W_tilde[j] + b[i] + b_tilde[j] - np.log(x_ij)
            coef = weight * diff
            total_loss += 0.5 * weight * diff * diff

            grad_W_i = coef * W_tilde[j]
            grad_W_tilde_j = coef * W[i]
            W[i] -= lr * grad_W_i
            W_tilde[j] -= lr * grad_W_tilde_j
            b[i] -= lr * coef
            b_tilde[j] -= lr * coef

        print(f"Epoch {epoch + 1}/{epochs} loss={total_loss / len(pairs):.4f}")

    return W + W_tilde

In [111]:
word_to_id, id_to_word, pair_counts = build_cooccurrence(
    docs,
    window=5,
    max_vocab=5000,
    min_count=5,
)

Vocab size: 5000
Co-occurrence pairs: 2816301


In [112]:
glove_vectors = glove_train(
    word_to_id,
    pair_counts,
    dim=50,
    epochs=10,
    lr=0.03,
    seed=42,
)

Epoch 1/10 loss=0.0325
Epoch 2/10 loss=0.0199
Epoch 3/10 loss=0.0168
Epoch 4/10 loss=0.0154
Epoch 5/10 loss=0.0144
Epoch 6/10 loss=0.0136
Epoch 7/10 loss=0.0129
Epoch 8/10 loss=0.0123
Epoch 9/10 loss=0.0117
Epoch 10/10 loss=0.0113


In [113]:
def nearest_words(word, word_to_id, id_to_word, vectors, topk=10):
    word = clean_review(word)
    if word not in word_to_id:
        raise ValueError(f"'{word}' is not in the vocabulary")

    target_idx = word_to_id[word]
    norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9
    vectors_norm = vectors / norms
    similarities = vectors_norm @ vectors_norm[target_idx]
    order = np.argsort(-similarities)

    results = []
    for idx in order:
        idx = int(idx)
        if idx == target_idx:
            continue
        results.append((id_to_word[idx], float(similarities[idx])))
        if len(results) == topk:
            break

    return results


nearest_words("discover", word_to_id, id_to_word, glove_vectors, topk=10)

[('needless', 0.6461168971548693),
 ('desperately', 0.6165491594708331),
 ('witness', 0.5640629041041457),
 ('attached', 0.5402884415421024),
 ('manage', 0.532922899276784),
 ('replace', 0.527547295917006),
 ('occur', 0.5208507198226683),
 ('randomly', 0.5153184873153801),
 ('warn', 0.5103457995032432),
 ('wanna', 0.5079349334117398)]

## FastText: Subwords of a Word

In [ ]:

def char_ngrams(word, n_min=3, n_max=6):
    wrapped = f"<{word}>"
    grams = {wrapped}
    for n in range(n_min, n_max + 1):
        for i in range(len(wrapped) - n + 1):
            grams.add(wrapped[i:i + n])
    return grams


def build_ngram_table_from_word_vectors(word_to_id, vectors):
    """Simple FastText-style table: each n-gram gets the average of words containing it."""
    sums = {}
    counts = Counter()

    for word, idx in word_to_id.items():
        for gram in char_ngrams(word):
            if gram not in sums:
                sums[gram] = np.zeros(vectors.shape[1])
            sums[gram] += vectors[idx]
            counts[gram] += 1

    return {gram: sums[gram] / counts[gram] for gram in sums}


def fasttext_vector(word, ngram_table):
    grams = char_ngrams(word)
    vecs = [ngram_table[g] for g in grams if g in ngram_table]
    if not vecs:
        return None
    return np.mean(vecs, axis=0)


def nearest_fasttext(word, vocabulary, ngram_table, topk=10):
    target = fasttext_vector(word, ngram_table)
    if target is None:
        raise ValueError(f"No known character n-grams for '{word}'")

    results = []
    target_norm = np.linalg.norm(target) + 1e-9
    for candidate in vocabulary:
        if candidate == word:
            continue
        vec = fasttext_vector(candidate, ngram_table)
        if vec is None:
            continue
        score = float((target @ vec) / (target_norm * (np.linalg.norm(vec) + 1e-9)))
        results.append((candidate, score))

    return sorted(results, key=lambda x: x[1], reverse=True)[:topk]



In [ ]:
ngram_table = build_ngram_table_from_word_vectors(word_to_id, glove_vectors)
vocabulary = list(word_to_id.keys())

nearest_fasttext("movie", vocabulary, ngram_table, topk=10)